# AfriHuBERT Fine-Tuning on Colab - Tshivenda (ven)

Third distinct ASR model family for the comparison (Wav2Vec2 = contrastive
self-supervision, Whisper = weakly-supervised encoder-decoder, HuBERT =
masked cluster-prediction self-supervision). Base checkpoint
`ajesujoba/AfriHuBERT` - confirmed to cover Tshivenda ('ven' in its 1240
language tags) and loads via standard `transformers` (`HubertForCTC`), no
extra toolkit needed.

Same bootstrap pattern as `colab_stage1_wav2vec2_ven.ipynb` - clone, dependencies,
regenerate data, train with Drive-backed checkpoints, evaluate.

**Before you start (one-time)**: same prerequisites as `colab_stage1_wav2vec2_ven.ipynb`
- GPU runtime, HF token, ANV gate accepted, `dev` up to date.

## 1. GPU check

In [ ]:
!nvidia-smi -L

## 2. Clone the repo + install dependencies

In [ ]:
!git clone -b dev https://github.com/Khotso-Bore/MultilingualASR.git
%cd MultilingualASR
!pip install -q -r requirements.txt

## 3. Hugging Face login (needed for the gated ANV dataset)

In [ ]:
from huggingface_hub import login
login()

## 4. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/multilingualasr"
OUTPUT_DIR = f"{DRIVE_ROOT}/hubert-ven-stage1"
PREDS_DIR = f"{DRIVE_ROOT}/preds-hubert-stage1"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 5. Regenerate the processed data on this runtime

Skip this cell if you already ran `colab_stage1_wav2vec2_ven.ipynb` in the same session - the data is identical, no need to redownload.

In [ ]:
!python src/preprocessing/preprocess_nchlt_ven.py
!python src/preprocessing/preprocess_anv_ven.py

## 6. AfriHuBERT training

Same recipe as Stage 1 Wav2Vec2 training, model swapped to `HubertForCTC` / `ajesujoba/AfriHuBERT`. Resumes automatically from the newest Drive checkpoint on reconnect.

In [ ]:
import numpy as np
import torch
from dataclasses import dataclass
from typing import Dict, List, Union
from datasets import load_dataset, Audio, Features, Value
from transformers import (
    HubertForCTC, Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)
from transformers.trainer_utils import get_last_checkpoint
from jiwer import wer, cer

BASE_CHECKPOINT = "ajesujoba/AfriHuBERT"
DATA = "dataset/processed"
features = Features({"audio": Audio(sampling_rate=16000), "transcript": Value("string")})
dataset_dict = load_dataset("csv", data_files={
    "train": [f"{DATA}/nchlt_ven/train.csv", f"{DATA}/anv_ven/train.csv"],
    "dev": [f"{DATA}/nchlt_ven/validation.csv", f"{DATA}/anv_ven/dev.csv"],
}, features=features)

tokenizer = Wav2Vec2CTCTokenizer.from_pretrained("tokenizers/ven")
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1, sampling_rate=16000, padding_value=0.0,
    do_normalize=True, return_attention_mask=True)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

def prepare_dataset(batch):
    samples = batch["audio"].get_all_samples()
    array = samples.data.numpy().squeeze()
    batch["input_values"] = processor(array, sampling_rate=16000).input_values[0]
    batch["labels"] = processor.tokenizer(batch["transcript"]).input_ids
    return batch

dataset_dict = dataset_dict.map(prepare_dataset, remove_columns=dataset_dict["train"].column_names)

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    def __call__(self, feats):
        inputs = [{"input_values": f["input_values"]} for f in feats]
        labels = [{"input_ids": f["labels"]} for f in feats]
        batch = self.processor.pad(inputs, padding=True, return_tensors="pt")
        labels_batch = self.processor.pad(labels=labels, padding=True, return_tensors="pt")
        batch["labels"] = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        return batch

def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    return {"wer": wer(label_str, pred_str), "cer": cer(label_str, pred_str)}

model = HubertForCTC.from_pretrained(
    BASE_CHECKPOINT,
    ctc_loss_reduction="mean", ctc_zero_infinity=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ignore_mismatched_sizes=True,
)
model.freeze_feature_encoder()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,   # T4 16GB: drop to 4 on OOM
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=1e-4,
    warmup_ratio=0.1,
    num_train_epochs=10,
    fp16=True,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    push_to_hub=False,
    report_to=[],
)

trainer = Trainer(
    model=model,
    data_collator=DataCollatorCTCWithPadding(processor=processor),
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["dev"],
    processing_class=processor.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

last_ckpt = get_last_checkpoint(OUTPUT_DIR)
print("resuming from:", last_ckpt or "scratch")
trainer.train(resume_from_checkpoint=last_ckpt)

trainer.save_model(f"{OUTPUT_DIR}/final")
processor.save_pretrained(f"{OUTPUT_DIR}/final")
print(f"saved -> {OUTPUT_DIR}/final")

## 7. Evaluate (comparable to the zero-shot and Wav2Vec2 tables)

In [ ]:
!python src/asr/evaluate_wav2vec2_ven.py --checkpoint {OUTPUT_DIR}/final --save-predictions {PREDS_DIR}